In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

def limpa_time(c):
    c = F.when(F.lower(c).isin("atlético-pr", "atletico-pr"), F.lit("Athletico-PR")).otherwise(c)
    c = F.regexp_replace(c, r"(?i)-\s*(FC|SC|EC)\s*$", "")   # remove -FC/-SC/-EC no final
    c = F.regexp_replace(c, r"(?i)\s+(FC|SC|EC)\s*$", "")   # remove " FC"/" SC"/" EC" no final
    return F.trim(F.regexp_replace(c, r"\s+", " "))

# 1) Brasileirão com DQ só nas duas colunas
(
  spark.table("bronze.brasileirao")
    .withColumn("time_mandante", limpa_time(F.col("time_mandante")))
    .withColumn("time_visitante", limpa_time(F.col("time_visitante")))
    .write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver.brasileirao")
)

# 2) Pass-through (sem ajustes)
for t in ["times", "estadio"]:
    (
      spark.table(f"bronze.{t}")
        .write.format("delta").mode("overwrite").option("overwriteSchema","true")
        .saveAsTable(f"silver.{t}")
    )